# Sesión 1 — Demo: Prompting y Embeddings con Gemini

Notebook de demostración para la **Sesión 1** del curso de IA Generativa (ISDI MDA).

**Contenido:**
1. Primera llamada a la API de Gemini — resumir, clasificar y responder una review
2. Embeddings: representación vectorial del texto
3. Proyección 2D: visualizar la estructura semántica de las reviews

**Dataset**: Amazon Product Reviews — Electronics (muestra de 50 reviews)

In [ ]:
%pip install -q google-genai pandas scikit-learn matplotlib

In [ ]:
from google import genai
import pandas as pd
import numpy as np

# Configurar la API key desde los Secrets de Colab
# Panel lateral → icono de llave → añadir GOOGLE_API_KEY
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

client = genai.Client(api_key=GOOGLE_API_KEY)

MODEL = 'gemini-2.5-flash'
EMBEDDING_MODEL = 'gemini-embedding-001'

print('✓ Gemini API configurada correctamente')

In [ ]:
import pandas as pd

# Dataset local — 50 reviews de Amazon Electronics curadas para la demo
DATA_URL = 'https://raw.githubusercontent.com/ber2/isdi-gen-ai/refs/heads/main/sesion1/amazon_electronics_50.csv'

try:
    sample = pd.read_csv(DATA_URL)
    print('✓ Dataset cargado desde GitHub')
except Exception:
    sample = pd.read_csv('sesion1/data/amazon_electronics_50.csv')
    print('✓ Dataset cargado desde fichero local')

print(f'Reviews: {len(sample)}')
print(f'Columnas: {list(sample.columns)}')
print(f'Distribución de ratings:\n{sample["rating"].value_counts().sort_index()}')

## Parte 1: Primera llamada a la API de Gemini

Vamos a tomar una review real del dataset y pedirle al modelo tres cosas:
1. Que la **resuma** en una frase en español
2. Que la **clasifique** como positiva, negativa o mixta
3. Que **proponga una respuesta** de atención al cliente

In [ ]:
# Elegir una review del sample
review = sample.iloc[0]

print(f'Producto (ASIN): {review["parent_asin"]}')
print(f'Rating: {review["rating"]} ★')
print(f'Título: {review["title"]}')
print(f'\nTexto de la review:')
print('-' * 60)
print(review['text'])

In [ ]:
# Tarea 1: Resumir la review
prompt_resumen = f"""Resume la siguiente review de cliente en una sola frase en español.
Sé conciso y capta lo más importante.

Review:
{review['text']}
"""

respuesta = client.models.generate_content(model=MODEL, contents=prompt_resumen)
print('RESUMEN:')
print(respuesta.text)

In [ ]:
# Tarea 2: Clasificar la review
prompt_clasificacion = f"""Eres un analista de experiencia del cliente para una tienda de electrónica.

Analiza la siguiente review y responde en JSON con exactamente estos campos:
- "clasificacion": "positiva", "negativa" o "mixta"
- "razon": explicación breve (máximo 25 palabras)
- "puntos_fuertes": lista de aspectos positivos mencionados
- "puntos_debiles": lista de aspectos negativos mencionados

Review:
{review['text']}

Responde SOLO con el JSON, sin texto adicional.
"""

respuesta = client.models.generate_content(model=MODEL, contents=prompt_clasificacion)
print('CLASIFICACIÓN:')
print(respuesta.text)

In [ ]:
# Tarea 3: Proponer una respuesta de atención al cliente
prompt_respuesta = f"""Eres el responsable de atención al cliente de una tienda de electrónica online.
Tu tono es profesional, empático y constructivo.

Un cliente ha dejado esta review:
{review['text']}

Redacta una respuesta pública de no más de 3 frases en español que:
1. Agradezca el feedback
2. Reconozca cualquier punto negativo si lo hay
3. Refuerce los aspectos positivos o explique cómo mejorar la experiencia
"""

respuesta = client.models.generate_content(model=MODEL, contents=prompt_respuesta)
print('RESPUESTA AL CLIENTE:')
print(respuesta.text)

## Parte 2: Embeddings y Espacios Semánticos

Un **embedding** convierte texto en un vector de números (768 dimensiones).
Textos con significado similar producen vectores cercanos en ese espacio.

Vamos a:
1. Generar embeddings para la muestra de 50 reviews
2. Reducir a 2 dimensiones con PCA
3. Visualizar si los reviews positivos y negativos se agrupan

In [ ]:
# Demostración: similitud semántica entre frases
import numpy as np

def get_embedding(text):
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text[:2000]
    )
    return np.array(result.embeddings[0].values)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Comparar frases semánticamente similares vs. distintas
frases = [
    'batería de larga duración',
    'autonomía de 30 horas',
    'entrega rápida del pedido'
]

embeddings_frases = [get_embedding(f) for f in frases]

print(f'Similitud "{frases[0]}" ≈ "{frases[1]}": '
      f'{cosine_similarity(embeddings_frases[0], embeddings_frases[1]):.3f}')
print(f'Similitud "{frases[0]}" ≈ "{frases[2]}": '
      f'{cosine_similarity(embeddings_frases[0], embeddings_frases[2]):.3f}')

In [ ]:
import time

print('Generando embeddings para 50 reviews (puede tardar ~1 minuto)...')
embeddings = []
for i, row in enumerate(sample.itertuples()):
    emb = get_embedding(row.text)
    embeddings.append(emb)
    if (i + 1) % 10 == 0:
        print(f'  {i + 1}/50 reviews procesadas')
    time.sleep(0.1)  # respetar el rate limit

X = np.array(embeddings)
print(f'\nMatriz de embeddings: {X.shape}  ({X.shape[0]} reviews × {X.shape[1]} dimensiones)')

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Reducir a 2 dimensiones
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)

varianza_explicada = pca.explained_variance_ratio_.sum() * 100
print(f'Varianza explicada por los 2 primeros componentes: {varianza_explicada:.1f}%')

# Visualizar coloreando por rating
fig, ax = plt.subplots(figsize=(10, 7))

ratings = sample['rating'].values
scatter = ax.scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=ratings, cmap='RdYlGn',
    vmin=1, vmax=5,
    alpha=0.8, s=80, edgecolors='white', linewidths=0.5
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Rating (estrellas)', fontsize=11)
cbar.set_ticks([1, 2, 3, 4, 5])

ax.set_title('Reviews de electrónica — proyección 2D de embeddings', fontsize=13, pad=15)
ax.set_xlabel(f'Componente principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=10)
ax.set_ylabel(f'Componente principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nObserva: ¿Se agrupan las reviews positivas (verde) y negativas (rojo)?')
print('Esto es la base del RAG — buscar reviews similares por significado, no por palabras exactas.')